# NLP Interview Study Guide

A self-contained reference: each section has **interview notes**, a **manual implementation**, and the **library equivalent**.

| # | Concept | Core idea | Freq |
|---|---|---|---|
| 1 | Tokenization | Split text into units (char / word / subword) | ★★★★☆ |
| 2 | Bag of Words | Count-vector representation | ★★★★★ |
| 3 | TF-IDF | Reweight BoW by how surprising a word is | ★★★★★ |
| 4 | N-grams | Contiguous token sequences | ★★★★☆ |
| 5 | Stemming & Lemmatization | Normalize word forms | ★★★☆☆ |
| 6 | BPE | Subword tokenization via greedy merges | ★★★★☆ |
| 7 | Cosine Similarity | Angle-based vector similarity | ★★★★★ |
| 8 | Edit Distance | Min edits to convert string A to B | ★★★★☆ |
| 9 | Word Embeddings | Dense semantic vectors; Word2Vec / GloVe | ★★★★★ |
| 10 | Scaled Dot-Product Attention | Q/K/V attention; backbone of transformers | ★★★★★ |
| 11 | BLEU Score | N-gram precision for evaluating generation | ★★★☆☆ |

**Install if needed:** `pip install nltk gensim tiktoken`  
**Assumed installed:** `numpy`, `pandas`, `scikit-learn`, `torch`

In [1]:
import numpy as np
import pandas as pd
import re, math
from collections import Counter, defaultdict

import nltk
for pkg in ['punkt', 'punkt_tab', 'wordnet', 'stopwords']:
    nltk.download(pkg, quiet=True)

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Shared corpus used throughout — semantic groupings make examples clear
DOCS = [
    'the cat sat on the mat',
    'the dog sat on the log',
    'cats and dogs are good animals',
    'the stock market crashed today',
]
print('Setup complete')
print(f'Corpus: {DOCS}')

Setup complete
Corpus: ['the cat sat on the mat', 'the dog sat on the log', 'cats and dogs are good animals', 'the stock market crashed today']


---
## 1. Tokenization

Split raw text into discrete units (tokens) before any further processing.

| Type | Unit | Example |
|---|---|---|
| Character | single char | `['h','e','l','l','o']` |
| Word | whitespace / regex split | `['hello', 'world']` |
| Subword | learned merge rules (BPE) | `['un', 'believ', 'able']` |

**Interview notes**
- Word tokenization must handle punctuation (naive `.split()` keeps `'world!'`).
- Subword tokenization (BPE, WordPiece, SentencePiece) is used in all modern LLMs — it balances vocabulary size vs OOV coverage.
- NLTK `word_tokenize` uses Punkt (unsupervised sentence boundary + word tokenizer).
- Know the difference between `word_tokenize` and `wordpunct_tokenize` (the latter splits on ALL punctuation).

In [2]:
text = 'NLP is fascinating! Dr. Smith tokenizes text into pieces.'

# 1. Character tokenization
char_tokens = list(text)
print('Char tokens (first 12):', char_tokens[:12])

# 2. Naive whitespace split — keeps punctuation attached
naive = text.lower().split()
print('\nNaive split:', naive)

# 3. Regex word tokenization — strips punctuation cleanly
word_tokens = re.findall(r'\b[a-z]+\b', text.lower())
print('Regex words:', word_tokens)

# 4. Sentence tokenization
sentences = re.split(r'(?<=[.!?])\s+', text)   # lookbehind for end-punctuation
print('\nSentences:', sentences)

Char tokens (first 12): ['N', 'L', 'P', ' ', 'i', 's', ' ', 'f', 'a', 's', 'c', 'i']

Naive split: ['nlp', 'is', 'fascinating!', 'dr.', 'smith', 'tokenizes', 'text', 'into', 'pieces.']
Regex words: ['nlp', 'is', 'fascinating', 'dr', 'smith', 'tokenizes', 'text', 'into', 'pieces']

Sentences: ['NLP is fascinating!', 'Dr.', 'Smith tokenizes text into pieces.']


In [3]:
from nltk.tokenize import word_tokenize, sent_tokenize, wordpunct_tokenize

text = 'NLP is fascinating! Dr. Smith tokenizes text into pieces.'

print('word_tokenize:     ', word_tokenize(text))
print('sent_tokenize:     ', sent_tokenize(text))
print('wordpunct_tokenize:', wordpunct_tokenize(text))
print('(wordpunct splits on ALL punctuation, including apostrophes)')

# Subword tokenization via tiktoken (GPT-4 BPE tokenizer)
try:
    import tiktoken
    enc = tiktoken.get_encoding('cl100k_base')
    ids = enc.encode(text)
    pieces = [enc.decode([t]) for t in ids]
    print(f'\ntiktoken BPE ({enc.n_vocab:,} vocab):')
    print('  IDs:    ', ids)
    print('  Tokens: ', pieces)
except ImportError:
    print('\n(tiktoken not installed — pip install tiktoken)')

word_tokenize:      ['NLP', 'is', 'fascinating', '!', 'Dr.', 'Smith', 'tokenizes', 'text', 'into', 'pieces', '.']
sent_tokenize:      ['NLP is fascinating!', 'Dr. Smith tokenizes text into pieces.']
wordpunct_tokenize: ['NLP', 'is', 'fascinating', '!', 'Dr', '.', 'Smith', 'tokenizes', 'text', 'into', 'pieces', '.']
(wordpunct splits on ALL punctuation, including apostrophes)

tiktoken BPE (100,277 vocab):
  IDs:     [45, 12852, 374, 27387, 0, 2999, 13, 9259, 4037, 4861, 1495, 1139, 9863, 13]
  Tokens:  ['N', 'LP', ' is', ' fascinating', '!', ' Dr', '.', ' Smith', ' token', 'izes', ' text', ' into', ' pieces', '.']


---
## 2. Bag of Words (BoW)

Represent each document as a **count vector** over a fixed vocabulary.

`bow[d][w] = count of word w in document d`

Properties: sparse, ignores word order and grammar, simple baseline.

**Interview notes**
- Always asked as warm-up: "how would you represent text numerically?"
- Know its limitations: `bank` (finance) == `bank` (river), word order lost.
- `CountVectorizer` has `max_features`, `min_df`, `max_df` to filter the vocabulary.
- Binary BoW: cap counts at 1 (`binary=True`) — presence not frequency.

In [4]:
# Build vocab and count matrix from scratch
tokenized = [doc.lower().split() for doc in DOCS]
vocab = sorted(set(tok for tokens in tokenized for tok in tokens))
w2i = {w: i for i, w in enumerate(vocab)}
print('Vocabulary:', vocab)
print(f'Vocab size: {len(vocab)}')

bow_matrix = np.zeros((len(DOCS), len(vocab)), dtype=int)
for d_idx, tokens in enumerate(tokenized):
    for tok in tokens:
        bow_matrix[d_idx, w2i[tok]] += 1

df_bow = pd.DataFrame(bow_matrix, columns=vocab,
                      index=[f'doc{i}' for i in range(len(DOCS))])
display(df_bow)

Vocabulary: ['and', 'animals', 'are', 'cat', 'cats', 'crashed', 'dog', 'dogs', 'good', 'log', 'market', 'mat', 'on', 'sat', 'stock', 'the', 'today']
Vocab size: 17


,and,animals,are,cat,cats,crashed,dog,dogs,good,log,market,mat,on,sat,stock,the,today
doc0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,2,0
doc1,0,0,0,0,0,0,1,0,0,1,0,0,1,1,0,2,0
doc2,1,1,1,0,1,0,0,1,1,0,0,0,0,0,0,0,0
doc3,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,1,1


In [5]:
cv = CountVectorizer()
X_bow = cv.fit_transform(DOCS)

df_bow_sk = pd.DataFrame(
    X_bow.toarray(),
    columns=cv.get_feature_names_out(),
    index=[f'doc{i}' for i in range(len(DOCS))]
)
print('sklearn CountVectorizer:')
display(df_bow_sk)

# Useful CountVectorizer arguments to know
cv2 = CountVectorizer(
    max_features=10,   # keep only top-10 by term frequency
    min_df=2,          # ignore terms that appear in < 2 documents
    binary=True,       # presence (1/0) instead of counts
    stop_words='english',  # drop common words ('the', 'and', ...)
)
print('With max_features=10, min_df=2, binary=True:')
print('Features:', cv2.fit(DOCS).get_feature_names_out())

sklearn CountVectorizer:


,and,animals,are,cat,cats,crashed,dog,dogs,good,log,market,mat,on,sat,stock,the,today
doc0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,2,0
doc1,0,0,0,0,0,0,1,0,0,1,0,0,1,1,0,2,0
doc2,1,1,1,0,1,0,0,1,1,0,0,0,0,0,0,0,0
doc3,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,1,1


With max_features=10, min_df=2, binary=True:
Features: ['sat']


---
## 3. TF-IDF — Term Frequency × Inverse Document Frequency

Downweight words that appear everywhere (low information) and upweight words that are distinctive to a document.

```
TF(t, d)  = count(t, d) / len(d)
IDF(t)    = log( (N + 1) / (df(t) + 1) ) + 1     # sklearn smoothed
TF-IDF    = TF × IDF
```
Then L2-normalize each document row (so dot product == cosine similarity).

**Interview notes**
- Classic interview question: implement TF-IDF from scratch.
- Smoothing (`+1` in numerator and denominator) prevents division by zero and avoids zero IDF for words in all docs.
- L2 normalization is done by sklearn by default — controls for document length.
- Common gotcha: the IDF formula varies between textbooks and libraries.

In [6]:
def compute_tfidf(docs):
    tokenized = [doc.lower().split() for doc in docs]
    N = len(tokenized)
    vocab = sorted(set(t for tokens in tokenized for t in tokens))
    w2i = {w: i for i, w in enumerate(vocab)}

    # Document frequency: how many docs contain each term
    df = np.zeros(len(vocab))
    for tokens in tokenized:
        for t in set(tokens):
            df[w2i[t]] += 1

    # IDF with sklearn's smoothing: log((N+1)/(df+1)) + 1
    idf = np.log((N + 1) / (df + 1)) + 1

    # Build TF-IDF matrix
    matrix = np.zeros((N, len(vocab)))
    for d_idx, tokens in enumerate(tokenized):
        counts = Counter(tokens)
        for t, cnt in counts.items():
            tf = cnt / len(tokens)
            matrix[d_idx, w2i[t]] = tf * idf[w2i[t]]

    # L2 normalize each document row (sklearn does this by default)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1
    matrix /= norms

    return pd.DataFrame(matrix.round(4), columns=vocab,
                        index=[f'doc{i}' for i in range(N)])

print('Manual TF-IDF:')
display(compute_tfidf(DOCS))
print()
print('Observation: "crashed", "market", "stock" score high only in doc3 (the finance doc)')

Manual TF-IDF:


,and,animals,are,cat,cats,crashed,dog,dogs,good,log,market,mat,on,sat,stock,the,today
doc0,0.0000,0.0000,0.0000,0.453,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,0.453,0.3572,0.3572,0.0000,0.5783,0.0000
doc1,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.453,0.0000,0.0000,0.453,0.0000,0.000,0.3572,0.3572,0.0000,0.5783,0.0000
doc2,0.4082,0.4082,0.4082,0.000,0.4082,0.0000,0.000,0.4082,0.4082,0.000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000
doc3,0.0000,0.0000,0.0000,0.000,0.0000,0.4763,0.000,0.0000,0.0000,0.000,0.4763,0.000,0.0000,0.0000,0.4763,0.3040,0.4763



Observation: "crashed", "market", "stock" score high only in doc3 (the finance doc)


In [7]:
tfidf = TfidfVectorizer()   # same formula as our manual version
X_tfidf = tfidf.fit_transform(DOCS)

df_tfidf = pd.DataFrame(
    X_tfidf.toarray().round(4),
    columns=tfidf.get_feature_names_out(),
    index=[f'doc{i}' for i in range(len(DOCS))]
)
print('sklearn TfidfVectorizer (should match manual above):')
display(df_tfidf)

print('IDF values for each term:')
idf_df = pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out()).sort_values(ascending=False)
print(idf_df.round(3).to_string())

sklearn TfidfVectorizer (should match manual above):


,and,animals,are,cat,cats,crashed,dog,dogs,good,log,market,mat,on,sat,stock,the,today
doc0,0.0000,0.0000,0.0000,0.453,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,0.453,0.3572,0.3572,0.0000,0.5783,0.0000
doc1,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.453,0.0000,0.0000,0.453,0.0000,0.000,0.3572,0.3572,0.0000,0.5783,0.0000
doc2,0.4082,0.4082,0.4082,0.000,0.4082,0.0000,0.000,0.4082,0.4082,0.000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000
doc3,0.0000,0.0000,0.0000,0.000,0.0000,0.4763,0.000,0.0000,0.0000,0.000,0.4763,0.000,0.0000,0.0000,0.4763,0.3040,0.4763


IDF values for each term:
and        1.916
dogs       1.916
stock      1.916
mat        1.916
market     1.916
log        1.916
animals    1.916
good       1.916
dog        1.916
crashed    1.916
cats       1.916
cat        1.916
are        1.916
today      1.916
on         1.511
sat        1.511
the        1.223


---
## 4. N-grams

An n-gram is a contiguous sequence of `n` tokens.

- **Unigram** (n=1): `['cat', 'sat']` — individual words
- **Bigram** (n=2): `[('cat','sat'), ('sat','on')]` — word pairs
- **Trigram** (n=3): `[('cat','sat','on'), ...]` — triples

N-grams capture local word order and phrases — something BoW misses.

**Interview notes**
- Character n-grams are used for language detection and spell checking.
- Higher n → sparser but more context. Trade-off: coverage vs specificity.
- Used in BLEU score evaluation (see Section 11).
- sklearn `CountVectorizer(ngram_range=(1,2))` adds both unigrams and bigrams.

In [8]:
def ngrams(tokens, n):
    'Generate all n-grams from a token list via sliding window.'
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

tokens = 'the cat sat on the mat'.split()

for n in [1, 2, 3]:
    grams = ngrams(tokens, n)
    label = ['unigram', 'bigram', 'trigram'][n-1]
    print(f'{label}s (n={n}): {grams}')

print()
# Character n-grams — useful for fuzzy matching and language detection
word = 'tokenize'
char_bigrams = [''.join(g) for g in ngrams(list(word), 2)]
print(f'Char bigrams of "{word}": {char_bigrams}')

# N-gram frequency distribution
from collections import Counter
bigram_counts = Counter(ngrams(tokens, 2))
print(f'\nBigram counts: {dict(bigram_counts)}')

unigrams (n=1): [('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',)]
bigrams (n=2): [('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat')]
trigrams (n=3): [('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat')]

Char bigrams of "tokenize": ['to', 'ok', 'ke', 'en', 'ni', 'iz', 'ze']

Bigram counts: {('the', 'cat'): 1, ('cat', 'sat'): 1, ('sat', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}


In [9]:
from nltk.util import ngrams as nltk_ngrams

tokens = 'the cat sat on the mat'.split()

print('NLTK bigrams:', list(nltk_ngrams(tokens, 2)))
print('NLTK trigrams (padded):',
      list(nltk_ngrams(tokens, 3, pad_left=True, pad_right=True,
                       left_pad_symbol='<PAD>', right_pad_symbol='<PAD>')))

# sklearn: extract multiple n-gram orders simultaneously
cv_ngram = CountVectorizer(ngram_range=(1, 2))  # unigrams + bigrams
X_ng = cv_ngram.fit_transform(DOCS)
print(f'\nsklearn CountVectorizer(ngram_range=(1,2)): {X_ng.shape[1]} features')
print('Features:', cv_ngram.get_feature_names_out().tolist())

NLTK bigrams: [('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat')]


TypeError: pad_sequence() got an unexpected keyword argument 'pad_symbol'

---
## 5. Stemming & Lemmatization

Both reduce a word to its base form to group morphological variants.

| | Stemming | Lemmatization |
|---|---|---|
| Method | Heuristic suffix stripping | Dictionary lookup (WordNet) |
| Speed | Fast | Slower |
| Output | May not be a real word (`argument -> argu`) | Always a valid word (`better -> good`) |
| Needs POS? | No | Yes (for correct result) |
| Package | `nltk.stem.PorterStemmer` | `nltk.stem.WordNetLemmatizer` |

**Interview notes**
- Stemming is lossy and aggressive; lemmatization is linguistically correct.
- Use stemming for IR/search (speed matters). Use lemmatization when interpretability matters.
- `WordNetLemmatizer` needs the POS tag (default is noun). Passing `pos='v'` gives verb form.
- Neither is used in modern transformer-based NLP — subword tokenization handles morphology implicitly.

In [ ]:
# Simplified suffix-stripping stemmer (illustrates the concept)
# Real Porter Stemmer has ~60 rules across 5 phases — this is a toy version
def simple_stem(word):
    word = word.lower()
    for suffix in ['ization', 'ational', 'ness', 'ing', 'tion', 'ies', 'est', 'ed', 'er', 'ly', 's']:
        if word.endswith(suffix) and len(word) > len(suffix) + 2:
            return word[:-len(suffix)]
    return word

words = ['running', 'happiness', 'organization', 'generalization', 'easily', 'better', 'cats']
print(f'{"Word":<20} {"Simple stem":<20}')
print('-' * 40)
for w in words:
    print(f'{w:<20} {simple_stem(w):<20}')
print()
print('Caveat: naive rules over-stem ("organization" -> "organiz") and miss exceptions')

In [ ]:
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

ps  = PorterStemmer()
lem = WordNetLemmatizer()

words = ['running', 'happiness', 'organization', 'generalization', 'easily', 'better', 'cats']

print(f'{"Word":<20} {"PorterStem":<20} {"Lemma(n)":<20} {"Lemma(v)":<20}')
print('-' * 80)
for w in words:
    stem   = ps.stem(w)
    l_noun = lem.lemmatize(w, pos='n')  # interpret as noun
    l_verb = lem.lemmatize(w, pos='v')  # interpret as verb
    print(f'{w:<20} {stem:<20} {l_noun:<20} {l_verb:<20}')

print()
print('Key examples:')
print('  "better" -> stem="better"  but  lemma(adj)="good"  (requires pos="a")')
print('  "organization" -> stem="organ" (too aggressive)  lemma="organization" (correct)')
print('  lem.lemmatize("better", pos="a") =', lem.lemmatize('better', pos='a'))

---
## 6. BPE — Byte Pair Encoding

A **data-driven subword tokenization** algorithm. Balances vocabulary size and OOV coverage — the key issue with pure word tokenization.

**Algorithm:**
1. Initialize vocabulary as individual characters (+ end-of-word marker `</w>`)
2. Count frequency of every adjacent symbol pair across the corpus
3. Merge the most frequent pair → add the merged symbol to vocabulary
4. Repeat steps 2–3 until target vocabulary size is reached

**Why it matters:**
- `unhappiness` → `un` + `happiness` (handles prefixes/suffixes)
- `GPT-4` → two tokens (no OOV problem)
- Used in GPT-2/3/4 (tiktoken), BERT (WordPiece — similar), and most modern LLMs

**Interview notes**
- Be able to trace through 2–3 merge steps by hand.
- WordPiece (BERT) differs: merges based on likelihood ratio, not raw frequency.
- SentencePiece works on raw characters (language-agnostic, handles spaces as characters).

In [ ]:
def get_pairs(vocab):
    'Count all adjacent symbol pairs, weighted by word frequency.'
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_pair(pair, vocab):
    'Merge all occurrences of pair in the vocabulary.'
    bigram = ' '.join(pair)
    merged = ''.join(pair)
    return {word.replace(bigram, merged): freq for word, freq in vocab.items()}

# Initialize: split each word into characters + end-of-word marker
corpus = ['low', 'low', 'lower', 'newest', 'newest', 'newest', 'widest']
vocab = Counter()
for word in corpus:
    vocab[' '.join(list(word)) + ' </w>'] += 1

print('Initial character-level vocabulary:')
for word, freq in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f'  {freq}x  [{word}]')
print()

# Run BPE merges
for step in range(8):
    pairs = get_pairs(vocab)
    if not pairs:
        break
    best_pair, best_freq = pairs.most_common(1)[0]
    vocab = merge_pair(best_pair, vocab)
    merged = ''.join(best_pair)
    print(f'Step {step+1}: merge {best_pair[0]!r} + {best_pair[1]!r} -> {merged!r}  '
          f'(freq={best_freq})  vocab: {list(vocab.keys())}')

In [ ]:
# tiktoken: OpenAI's BPE tokenizer library (GPT-4 uses cl100k_base)
try:
    import tiktoken
    enc = tiktoken.get_encoding('cl100k_base')
    print(f'Tokenizer: {enc.name}  |  Vocab size: {enc.n_vocab:,}\n')

    examples = [
        'low lower newest widest',     # matches our manual corpus
        'tokenization',
        'unbelievably',
        '2024-01-15',
        'ChatGPT is fascinating!',
        'antidisestablishmentarianism', # long rare word -> many subwords
    ]
    for text in examples:
        ids    = enc.encode(text)
        pieces = [enc.decode([t]) for t in ids]
        print(f'{text!r:<35} -> {len(ids)} tokens: {pieces}')
except ImportError:
    print('tiktoken not installed. pip install tiktoken')
    print()
    print('Alternative: HuggingFace tokenizers library')
    print('  from tokenizers import Tokenizer')
    print('  from tokenizers.models import BPE')
    print('  tokenizer = Tokenizer.from_pretrained("gpt2")')

---
## 7. Cosine Similarity

Measure the **angle** between two vectors — not their magnitude. This makes it length-invariant (a 10-word doc and a 100-word doc can still be very similar).

```
cos(a, b) = (a · b) / (||a|| × ||b||)    range: [-1, 1]
```

For TF-IDF vectors (all non-negative), range is `[0, 1]`.

**Interview notes**
- Most common follow-up to BoW/TF-IDF: "how would you find similar documents?" → cosine similarity.
- **Euclidean distance** is sensitive to vector magnitude (long docs dominate). Cosine is not.
- `sklearn.cosine_similarity` returns a **similarity** (1 = identical). `scipy.spatial.distance.cosine` returns a **distance** (0 = identical). Common mistake!
- Used in: semantic search, recommendation systems, duplicate detection, clustering.

In [ ]:
def cosine_similarity(a, b):
    'cos(a, b) = dot(a, b) / (norm(a) * norm(b))'
    dot    = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

# Use TF-IDF vectors for meaningful similarity
tf = TfidfVectorizer()
X = tf.fit_transform(DOCS).toarray()

print('Documents:')
for i, doc in enumerate(DOCS):
    print(f'  doc{i}: {doc!r}')
print()

print(f'{"Pair":<20} {"Cosine Sim"}')
print('-' * 35)
for i, j in [(0,1), (0,2), (1,2), (0,3), (2,3)]:
    sim = cosine_similarity(X[i], X[j])
    print(f'doc{i} vs doc{j}           {sim:.4f}')
print()
print('doc0-doc1 (both cat/dog sentences) -> high similarity')
print('doc0-doc3 (animals vs finance)     -> low similarity')

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from scipy.spatial.distance import cosine as scipy_cosine  # returns DISTANCE

print('sklearn cosine_similarity (full 4x4 matrix):')
sim_matrix = sk_cosine(X)
display(pd.DataFrame(sim_matrix.round(4),
                     columns=[f'doc{i}' for i in range(len(DOCS))],
                     index=[f'doc{i}' for i in range(len(DOCS))]))

print('scipy cosine (returns DISTANCE = 1 - similarity):')
for i, j in [(0,1), (0,3)]:
    dist = scipy_cosine(X[i], X[j])
    print(f'  scipy_cosine(doc{i}, doc{j}) = {dist:.4f}  '
          f'(similarity = {1-dist:.4f})')
print()
print('IMPORTANT: sklearn returns similarity [0,1], scipy returns distance [0,2]!')

---
## 8. Edit Distance (Levenshtein Distance)

Minimum number of single-character **insert / delete / substitute** operations to transform string `s1` into `s2`.

Solved with **dynamic programming**:
```
dp[i][j] = edit_distance(s1[:i], s2[:j])

if s1[i-1] == s2[j-1]:  dp[i][j] = dp[i-1][j-1]
else:                    dp[i][j] = 1 + min(dp[i-1][j],   # delete
                                            dp[i][j-1],   # insert
                                            dp[i-1][j-1]) # substitute
```
**Time:** O(m·n)  **Space:** O(m·n) — can be reduced to O(min(m,n)) with rolling array.

**Interview notes**
- Very common coding interview question — know this cold.
- Variations: Damerau-Levenshtein adds **transposition** (ab→ba costs 1 not 2).
- Applications: spell checking, fuzzy string matching, DNA sequence alignment.
- `nltk.edit_distance` supports the `transpositions=True` flag.

In [ ]:
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    # dp[i][j] = min edits to convert s1[:i] -> s2[:j]
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i  # delete all of s1
    for j in range(n + 1): dp[0][j] = j  # insert all of s2

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]             # chars match, no cost
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],    # delete s1[i-1]
                    dp[i][j-1],    # insert s2[j-1]
                    dp[i-1][j-1],  # substitute s1[i-1] -> s2[j-1]
                )
    return dp[m][n], dp

s1, s2 = 'kitten', 'sitting'
dist, table = edit_distance(s1, s2)
print(f'edit_distance({s1!r}, {s2!r}) = {dist}')
print()
print('DP table (row = char of s1 incl. empty, col = char of s2 incl. empty):')
display(pd.DataFrame(table, index=list(' ' + s1), columns=list(' ' + s2)))

print()
pairs = [('cat', 'cats'), ('book', 'back'), ('sunday', 'saturday'), ('hello', 'hello'), ('', 'abc')]
for a, b in pairs:
    d, _ = edit_distance(a, b)
    print(f'  edit_distance({a!r}, {b!r}) = {d}')

In [ ]:
import nltk

pairs = [('kitten', 'sitting'), ('cat', 'cats'), ('book', 'back'), ('sunday', 'saturday')]
print('nltk.edit_distance (standard Levenshtein):')
for a, b in pairs:
    d = nltk.edit_distance(a, b)
    print(f'  ({a!r}, {b!r}) = {d}')

print()
print('Damerau-Levenshtein (transposition = swap two adjacent chars costs 1):')
for a, b in [('ab', 'ba'), ('ca', 'abc')]:
    std   = nltk.edit_distance(a, b)
    damlev = nltk.edit_distance(a, b, transpositions=True)
    print(f'  ({a!r}, {b!r})  standard={std}  with_transpositions={damlev}')

---
## 9. Word Embeddings

Map words to **dense, low-dimensional** vectors where semantically similar words are geometrically close.

**Word2Vec** (Mikolov 2013) — two training objectives:
- **Skip-gram**: given a center word, predict surrounding context words
- **CBOW**: given context words, predict the center word

**GloVe** (Pennington 2014): factorizes the co-occurrence matrix log-probabilities.

**Key properties:**
- `king - man + woman ≈ queen` (linear arithmetic in embedding space)
- Similar words cluster together (cats, dogs, animals)
- OOV words not handled — subword models (fastText, BERT) fix this

**Interview notes**
- Know the intuition: words that appear in similar contexts get similar vectors (distributional hypothesis).
- Skip-gram works better on rare words; CBOW is faster and better on frequent words.
- Modern approach: contextual embeddings (BERT, GPT) — same word gets different vector based on context (`bank` in finance vs river).
- Package: `gensim` for Word2Vec; `sentence-transformers` for sentence-level embeddings.

In [ ]:
# Co-occurrence matrix + SVD = GloVe-like static embeddings
# Core insight: words that co-occur frequently should have similar vectors

corpus = [
    ['I', 'love', 'cats'],
    ['I', 'love', 'dogs'],
    ['cats', 'and', 'dogs', 'are', 'animals'],
    ['animals', 'love', 'food'],
    ['cats', 'eat', 'food'],
    ['dogs', 'eat', 'food'],
]
vocab = sorted(set(w for sent in corpus for w in sent))
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

# Build symmetric co-occurrence matrix with window=1
cooc = np.zeros((V, V))
for sent in corpus:
    for i, word in enumerate(sent):
        for ctx in sent[max(0, i-1):i] + sent[i+1:min(len(sent), i+2)]:
            cooc[w2i[word], w2i[ctx]] += 1

print('Co-occurrence matrix (window=1):')
display(pd.DataFrame(cooc.astype(int), index=vocab, columns=vocab))

# SVD: decompose into dense low-dim embeddings
U, S, Vt = np.linalg.svd(cooc)
dim = 2
embeddings = U[:, :dim] * np.sqrt(S[:dim])   # absorb singular values into U

print(f'\n{dim}D embeddings (via SVD of co-occurrence matrix):')
for word, emb in zip(vocab, embeddings):
    print(f'  {word:<10} [{emb[0]:+.3f}, {emb[1]:+.3f}]')

def cosim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

print()
print('cosine_sim(cats, dogs)    =', round(cosim(embeddings[w2i['cats']], embeddings[w2i['dogs']]), 3))
print('cosine_sim(cats, food)    =', round(cosim(embeddings[w2i['cats']], embeddings[w2i['food']]), 3))
print('cosine_sim(cats, animals) =', round(cosim(embeddings[w2i['cats']], embeddings[w2i['animals']]), 3))

In [ ]:
try:
    from gensim.models import Word2Vec

    sentences = [
        ['I', 'love', 'cats'],
        ['I', 'love', 'dogs'],
        ['cats', 'and', 'dogs', 'are', 'animals'],
        ['animals', 'love', 'food'],
        ['cats', 'eat', 'food'],
        ['dogs', 'eat', 'food'],
        ['cats', 'are', 'cute', 'animals'],
        ['dogs', 'are', 'loyal', 'animals'],
    ]

    model = Word2Vec(
        sentences=sentences,
        vector_size=16,  # embedding dimension
        window=2,        # context window (words to left + right)
        min_count=1,     # include all words even if they appear once
        sg=1,            # 1=skip-gram, 0=CBOW
        epochs=300,
        seed=42,
    )

    print('Vocabulary:', sorted(model.wv.key_to_index.keys()))
    print()
    print("Vector for 'cats' (first 8 dims):", model.wv['cats'][:8].round(3))
    print()
    print("Most similar to 'cats':")
    for word, score in model.wv.most_similar('cats', topn=4):
        print(f'  {word:<12} similarity={score:.3f}')
    print()
    print("Most similar to 'food':")
    for word, score in model.wv.most_similar('food', topn=4):
        print(f'  {word:<12} similarity={score:.3f}')

except ImportError:
    print('gensim not installed. pip install gensim')
    print()
    print('Alternative: sentence-transformers (sentence-level, not word-level)')
    print('  from sentence_transformers import SentenceTransformer')
    print('  model = SentenceTransformer("all-MiniLM-L6-v2")')
    print('  vecs = model.encode(["cat", "dog", "stock market"])')

---
## 10. Scaled Dot-Product Attention

The core computation inside every transformer layer.

```
Attention(Q, K, V) = softmax( QKᵀ / √d_k ) · V
```

- **Q (Query)**: what the current position is looking for
- **K (Key)**: what each position offers
- **V (Value)**: what each position actually returns
- **√d_k scaling**: prevents dot products from growing large (→ vanishing gradients through softmax)

**Multi-head attention** runs `h` parallel attention heads, concatenates outputs, and projects:
```
MultiHead(Q,K,V) = Concat(head_1,...,head_h) · W_O
```

**Interview notes**
- "Why scale by √d_k?" — dot products grow as O(d_k), scaling keeps softmax in a sensitive range.
- **Causal masking** (decoder): set future positions to -∞ before softmax → 0 attention weight → can't see future tokens.
- Self-attention: Q=K=V come from the same sequence. Cross-attention (encoder-decoder): Q from decoder, K/V from encoder.
- Time complexity: O(n² · d) — quadratic in sequence length n. Main bottleneck for long contexts.

In [ ]:
def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))  # subtract max for numerical stability
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (seq_q, d_k)
    K: (seq_k, d_k)
    V: (seq_k, d_v)
    Returns: output (seq_q, d_v), attention_weights (seq_q, seq_k)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)   # (seq_q, seq_k)

    if mask is not None:
        scores = scores + mask * -1e9  # large negative -> ~0 after softmax

    weights = softmax(scores, axis=-1) # (seq_q, seq_k) — each row sums to 1
    output  = weights @ V              # (seq_q, d_v)
    return output, weights

np.random.seed(42)
seq_len, d_k, d_v = 5, 8, 8
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_v)

output, weights = scaled_dot_product_attention(Q, K, V)
print('Input shapes: Q, K, V =', Q.shape)
print('Output shape:', output.shape)
print()
print('Attention weights (each row sums to 1):')
labels = [f'pos{i}' for i in range(seq_len)]
display(pd.DataFrame(weights.round(3), index=labels, columns=labels))

# Causal mask: position i can only attend to j <= i (for autoregressive models like GPT)
causal_mask = np.triu(np.ones((seq_len, seq_len)), k=1)  # 1s in upper triangle
_, masked_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print('Causal masked weights (lower triangle only — no future peeking):')
display(pd.DataFrame(masked_weights.round(3), index=labels, columns=labels))

In [ ]:
try:
    import torch
    import torch.nn.functional as F
    import torch.nn as nn

    # Wrap arrays as tensors (add batch dim)
    Q_t = torch.tensor(Q, dtype=torch.float32).unsqueeze(0)  # (1, seq, d_k)
    K_t = torch.tensor(K, dtype=torch.float32).unsqueeze(0)
    V_t = torch.tensor(V, dtype=torch.float32).unsqueeze(0)

    # F.scaled_dot_product_attention (PyTorch >= 2.0)
    out_t = F.scaled_dot_product_attention(Q_t, K_t, V_t)  # (1, seq, d_v)
    print('F.scaled_dot_product_attention output (first row):')
    print(out_t[0, 0].detach().numpy().round(4))
    print('Manual output (first row):')
    print(output[0].round(4))
    print('Match:', np.allclose(out_t[0].detach().numpy(), output, atol=1e-5))
    print()

    # nn.MultiheadAttention — wraps multiple heads + projection matrices
    mha = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)
    mha_out, attn_weights = mha(Q_t, K_t, V_t)
    print('nn.MultiheadAttention (2 heads, embed_dim=8):')
    print('  output shape:      ', mha_out.shape)
    print('  attn_weights shape:', attn_weights.shape)

    # Causal mask with PyTorch
    causal = nn.Transformer.generate_square_subsequent_mask(seq_len)  # upper tri = -inf
    out_causal, _ = mha(Q_t, K_t, V_t, attn_mask=causal)
    print('  causal output shape:', out_causal.shape)

except ImportError:
    print('torch not installed. pip install torch')

---
## 11. BLEU Score — Bilingual Evaluation Understudy

Measures how much a **candidate** text (model output) overlaps with one or more **reference** texts.

```
BLEU = BP × exp( Σ w_n × log(p_n) )

p_n  = clipped n-gram precision (see below)
BP   = brevity penalty = min(1, exp(1 - r/c))   where c=candidate len, r=reference len
w_n  = weight per n-gram order (default 0.25 each for BLEU-4)
```

**Clipping**: cap each candidate n-gram count by its maximum count in any reference. Prevents score inflation from repetition (e.g., `the the the`).

**Interview notes**
- BLEU-4 (n up to 4) is the standard for machine translation benchmarks.
- Brevity penalty punishes candidates shorter than references.
- Limitations: ignores synonyms, word order flexibility, and semantics entirely.
- Alternatives: **ROUGE** (recall-oriented, used for summarization), **METEOR** (handles synonyms), **BERTScore** (embedding-based).

In [ ]:
def _ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def clipped_precision(candidate, references, n):
    'n-gram precision clipped by max count in any reference.'
    cand_counts = Counter(_ngrams(candidate, n))

    # Maximum this n-gram appears in any single reference
    max_ref = Counter()
    for ref in references:
        for gram, cnt in Counter(_ngrams(ref, n)).items():
            max_ref[gram] = max(max_ref[gram], cnt)

    # Clip: candidate count cannot exceed reference maximum
    clipped = sum(min(cnt, max_ref[gram]) for gram, cnt in cand_counts.items())
    total   = max(1, sum(cand_counts.values()))
    return clipped / total

def bleu(candidate, references, max_n=4):
    # Brevity penalty
    c = len(candidate)
    r = min(len(ref) for ref in references)   # closest reference length
    bp = 1.0 if c >= r else math.exp(1 - r / c)

    # Geometric mean of clipped n-gram precisions (in log-space)
    w = 1.0 / max_n
    log_avg = 0.0
    for n in range(1, max_n + 1):
        p_n = clipped_precision(candidate, references, n)
        if p_n == 0:
            return 0.0   # any zero precision -> BLEU = 0
        log_avg += w * math.log(p_n)
    return bp * math.exp(log_avg)

cand  = 'the cat sat on the mat'.split()
refs  = [
    'the cat is on the mat'.split(),
    'there is a cat on the mat'.split(),
]

print('Candidate: ', ' '.join(cand))
for i, r in enumerate(refs):
    print(f'Reference {i}: {" ".join(r)}')
print()

for n in range(1, 5):
    p = clipped_precision(cand, refs, n)
    print(f'{n}-gram clipped precision: {p:.4f}')

print(f'\nBLEU-4 (manual): {bleu(cand, refs):.4f}')

# Contrast with bad candidate
bad = 'dog runs fast in park'.split()
print(f'BLEU-4 (bad cand "{" ".join(bad)}"): {bleu(bad, refs):.4f}')

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction

cand  = 'the cat sat on the mat'.split()
refs  = [
    'the cat is on the mat'.split(),
    'there is a cat on the mat'.split(),
]

print('NLTK sentence_bleu:')
for n in range(1, 5):
    w = [1/n] * n + [0] * (4 - n)   # e.g. BLEU-2 = (0.5, 0.5, 0, 0)
    score = sentence_bleu(refs, cand, weights=w)
    print(f'  BLEU-{n}: {score:.4f}')

bleu4 = sentence_bleu(refs, cand)   # default: equal 0.25 weights up to n=4
print(f'\nBLEU-4 (nltk):   {bleu4:.4f}')
print(f'BLEU-4 (manual): {bleu(cand, refs):.4f}  <- should match')

# Smoothing: needed when candidate is short and some n-gram precisions are 0
smooth = SmoothingFunction().method1
short_cand = 'cat mat'.split()
print(f'\nShort candidate "{" ".join(short_cand)}":')
print(f'  BLEU-4 no smoothing: {sentence_bleu(refs, short_cand):.4f}   (= 0 because 4-gram precision is 0)')
print(f'  BLEU-4 smoothed:     {sentence_bleu(refs, short_cand, smoothing_function=smooth):.4f}')

# Perfect and terrible candidates
print(f'\nPerfect (matches ref 0): {sentence_bleu(refs, refs[0]):.4f}')
print(f'Terrible ("dog barks"):  {sentence_bleu(refs, ["dog", "barks"]):.4f}')